In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
import os

from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemini-3.1-flash-lite",
    model_provider="google-genai",
    api_key=os.getenv("GOOGLE_API_KEY"),
)


In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [4]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is exploring fictional or world-building details regarding "Lunapolis," the capital of the moon, specifically focusing on its environmental conditions, population demographics (cheese miners), and current political/labor climate.\n\n## SUMMARY\n\n*   **Location:** Lunapolis is established as the capital of the moon.\n*   **Environment:** Extremely high temperature variance (High: 120C, Low: -100C) with clear skies.\n*   **Demographics:** The city has a population of 100,000 "cheese miners."\n*   **Political Status:** There is internal instability; the cheese miners\' union is expected to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\n\nNone.\n\n## NEXT STEPS\n\nContinue responding to user inquiries regarding the fictional setting of Lunapolis or provide further world-building details as requested.', additional_kwargs={'lc_source': 'summarization'},

In [5]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

The user is exploring fictional or world-building details regarding "Lunapolis," the capital of the moon, specifically focusing on its environmental conditions, population demographics (cheese miners), and current political/labor climate.

## SUMMARY

*   **Location:** Lunapolis is established as the capital of the moon.
*   **Environment:** Extremely high temperature variance (High: 120C, Low: -100C) with clear skies.
*   **Demographics:** The city has a population of 100,000 "cheese miners."
*   **Political Status:** There is internal instability; the cheese miners' union is expected to strike due to dissatisfaction with the new president.

## ARTIFACTS

None.

## NEXT STEPS

Continue responding to user inquiries regarding the fictional setting of Lunapolis or provide further world-building details as requested.


## Trim/delete messages

In [6]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [7]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [8]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='05970f79-156d-41e4-9384-efad8eff6c79'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='1e157d6a-3ae5-4d9a-9f2e-3b82bed1f46a', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='23cc840a-6c17-4030-810c-3dfe7aef32a7'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='5ca03cc9-5e6d-4807-9b55-4db48006758b', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='5eeda7ea-0ae3-40d1-a348-658b13277f4a'),
              AIMessage(content=[{'type': 'text', 'text': 'To better assist you, I need to know

In [9]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': 'To better assist you, I need to know a little more about the device. Are you asking me to tell you the temperature, or are you checking if the device feels unusually hot?\n\n**As an AI, I cannot physically sense the temperature of your device.** However, temperature is a very important clue:\n\n1.  **If it feels hot to the touch:** \n    *   Unplug it immediately. \n    *   It may be overheating, which can cause a safety shutdown or indicate a hardware failure. Let it cool down completely for at least 30 minutes before trying anything else.\n\n2.  **If it feels normal (room temperature):**\n    *   **The "Hard Reset" (for most electronics):** Even if a device is plugged in, it might be in a "frozen" state. Try holding the power button down for a full **15–30 seconds** without letting go. Sometimes this forces a reboot that a quick press cannot trigger.\n    *   **Check the Power Source:** If possible, try plugging a different device (like a lamp or phone char